# Flusso completo: da HuggingFace a llama.cpp (GGUF)
## Modello: Mistral-7B-v0.1

Flusso:
1. Download del modello da HuggingFace
2. Caricamento e test con Transformers
3. Conversione in GGUF (f16)
4. Quantizzazione (Q4_K_M)
5. Inferenza con llama-cpp-python

---
## 0. Setup virtualenv

> **Esegui questi comandi nel terminale** (fuori da Jupyter) prima di avviare il notebook.

```bash
# Crea il virtualenv nella cartella del progetto
python3 -m venv .venv

# Attiva il virtualenv
source .venv/bin/activate          # macOS / Linux
# .venv\Scripts\activate           # Windows

# Installa il kernel Jupyter nel virtualenv
pip install ipykernel
python -m ipykernel install --user --name=quantizzazione --display-name "Python (quantizzazione)"

# Avvia Jupyter (solo se non usi vscode)
jupyter notebook
```

---
## 0b. Installazione dipendenze (nel virtualenv)

In [3]:
# Installa le librerie Python necessarie
import sys
print(f"Python in uso: {sys.executable}")
# Verifica che punti al virtualenv (.venv) prima di proseguire

!pip install transformers torch accelerate huggingface_hub llama-cpp-python
!pip install --upgrade requests

# ---------------------------------------------------------------------------
# PRE-REQUISITO: cmake deve essere installato sul sistema.
#
# macOS:   brew install cmake
# Linux:   sudo apt install cmake  (oppure: sudo dnf install cmake)
#
# Windows (decommentare e scegliere uno dei metodi):
#   Metodo 1 - winget (Windows 10/11, nessun tool extra):
#     winget install Kitware.CMake
#
#   Metodo 2 - Chocolatey:
#     choco install cmake --installargs 'ADD_CMAKE_TO_PATH=System'
#
#   Metodo 3 - Scoop:
#     scoop install cmake
#
#   Metodo 4 - installer grafico:
#     https://cmake.org/download/  → cmake-*-windows-x86_64.msi
# ---------------------------------------------------------------------------

# Clona e compila llama.cpp con CMake (le versioni recenti non supportano 'make' diretto)
import os
import platform
if not os.path.isdir('llama.cpp'):
    !git clone https://github.com/ggerganov/llama.cpp

# Determina il numero di core in modo cross-platform
import multiprocessing
cores: int = multiprocessing.cpu_count()

# Percorso del binario: su Windows finisce in build/bin/Release/
bin_path = (
    'llama.cpp/build/bin/Release/llama-quantize.exe'
    if platform.system() == 'Windows'
    else 'llama.cpp/build/bin/llama-quantize' 
)

if not os.path.isfile(bin_path):
    !cmake -S llama.cpp -B llama.cpp/build -DCMAKE_BUILD_TYPE=Release
    !cmake --build llama.cpp/build --config Release -j{cores}
    !pip install -r llama.cpp/requirements.txt 
else:
    print('llama.cpp già compilato, skip.')

Python in uso: c:\Users\A1618apulia\OneDrive - ITS Apulia Digital Maker\Desktop\GENERATIVEAI\.venv\Scripts\python.exe
  Using cached llama_cpp_python-0.3.36.tar.gz (76.6 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
Using cached diskcache-5.6.3-py3-none-any.whl (45 kB)
Failed to build llama-cpp-python


  error: subprocess-exited-with-error
  
  × Building wheel for llama-cpp-python (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [19 lines of output]
      *** scikit-build-core 1.1.0 using CMake 4.4.3 (wheel)
      *** Configuring CMake...
      loading initial cache file C:\Users\A1618A~1\AppData\Local\Temp\tmpnhl315rx\build\CMakeInit.txt
      -- Building for: NMake Makefiles
      CMake Error at CMakeLists.txt:3 (project):
        Running
      
         'nmake' '-?'
      
        failed with:
      
         no such file or directory
      
      
      CMake Error: CMAKE_C_COMPILER not set, after EnableLanguage
      CMake Error: CMAKE_CXX_COMPILER not set, after EnableLanguage
      -- Configuring incomplete, errors occurred!
      
      *** CMake configuration failed
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for llama-cpp-python
ERROR: Could not build wheels 

---
## 1. Download del modello da HuggingFace (se non ancora presente)

In [5]:
from huggingface_hub import snapshot_download
import os

MODEL_ID = "mistralai/Mistral-7B-v0.1"
MODEL_DIR = "./Mistral-7B-v0.1"

# MODEL_ID = "ibm-granite/granite-4.2-3b"
# MODEL_DIR = "./Granite-4.2-3B"

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "120"

snapshot_download(
    repo_id=MODEL_ID,
    local_dir=MODEL_DIR,
    ignore_patterns=["*.bin"],
)
print(f"Modello scaricato in: {MODEL_DIR}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Modello scaricato in: ./Mistral-7B-v0.1


---
## 2. Caricamento e test con Transformers (opzionale ma utile per verifica)

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_DIR = "./Mistral-7B-v0.1"
#MODEL_DIR = "./Granite-4.2-3B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.bfloat16, # Carica il modello in virgola mobile a 16 bit
    device_map="auto", # `device_map="auto"` splits it across available GPUs efficiently
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


FileNotFoundError: No such file or directory: ./Mistral-7B-v0.1\model-00001-of-00002.safetensors

In [28]:
prompt = "La quantizzazione di un modello LLM consiste in"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=100, do_sample=False)

print(f'\n\n RISPOSTA DEL MODELLO\n')
print(tokenizer.decode(output[0], skip_special_tokens=True))

# Mistral-7B-v0.1: Circa 4 minuti su M4 MAX
# RISPOSTA DEL MODELLO:
# La quantizzazione di un modello LLM consiste in una serie di passaggi che possono essere riassunti in tre fasi principali:
# 1. **Preparazione del modello**: In questa fase, si prepara il modello LLM per la quantizzazione. Questo può includere la conversione del modello in un formato compatibile con il framework di quantizzazione scelto, la rimozione di parti del modello non necessarie per la quantizzazione e la

# Granite-4.2-3B: Circa 4 secondi su M4 MAX

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.




 RISPOSTA DEL MODELLO

La quantizzazione di un modello LLM consiste in una serie di passaggi che possono essere riassunti in tre fasi principali:

1. **Preparazione del modello**: In questa fase, si prepara il modello LLM per la quantizzazione. Questo può includere la conversione del modello in un formato compatibile con il framework di quantizzazione scelto, la rimozione di parti del modello non necessarie per la quantizzazione e la


---
## 3. Conversione in GGUF (f16)

Usa lo script `convert_hf_to_gguf.py` incluso in llama.cpp.

In [13]:
GGUF_F16 = "./mistral-7b-v0.1-f16.gguf"

#outtype può essere f32, f16, bf16, q8_0 — usa f16 come punto di partenza.

!python llama.cpp/convert_hf_to_gguf.py \
    ./Mistral-7B-v0.1 \
    --outtype f16 \
    --outfile {GGUF_F16}

print(f"File GGUF f16 generato: {GGUF_F16}")

INFO:hf-to-gguf:Loading model: Mistral-7B-v0.1
INFO:hf-to-gguf:Model architecture: MistralForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00002.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00002.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> F16, shape = {4096, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.bfloat16 --> F32, shape = {4096}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.bfloat16 --> F16, shape = {14336, 4096}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.bfloat16 --> F16, shape = {4096, 14336}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.bfloat16 --> F16, shape = {4096, 14336}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.bfloat16 --> F32, shape = {4096}
INFO:hf-to-gguf:blk.0.att

---
## 4. Quantizzazione a Q4_K_M

Riduce il modello da ~14 GB a ~4 GB mantenendo una buona qualità.

In [21]:
GGUF_F16  = "./mistral-7b-v0.1-f16.gguf"
GGUF_Q4   = "./mistral-7b-v0.1-Q4_K_M.gguf"
QUANT_TYPE = "Q4_K_M"

# I binari compilati con CMake si trovano in llama.cpp/build/bin/
!./llama.cpp/build/bin/llama-quantize {GGUF_F16} {GGUF_Q4} {QUANT_TYPE}

print(f"File quantizzato ({QUANT_TYPE}) generato: {GGUF_Q4}")

ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library
ggml_metal_library_compile_all: compiled 'fa' library in 12.677 sec
ggml_metal_library_compile_all: compiled 'mul_mv' library in 0.891 sec
ggml_metal_library_compile_all: compiled 'mul_mm' library in 1.117 sec
ggml_metal_library_compile_all: compiled 'quantize' library in 0.456 sec
ggml_metal_library_compile_all: compiled 'softmax' library in 0.240 sec
ggml_metal_library_compile_all: compiled 'norm' library in 0.456 sec
ggml_metal_library_compile_all: compiled 'unary' library in 0.253 sec
ggml_metal_library_compile_all: compiled 'binbcast' library in 0.248 sec
ggml_metal_library_compile_all: compiled 'reduce' library in 0.260 sec
ggml_metal_library_compile_all: compiled 'tri' library in 0.241 sec
ggml_metal_library_compile_all: compiled 'ssm' library in 0.248 sec
ggml_metal_library_compile_all: compiled 'wkv' library in 0.250 sec
ggml_metal_library_compile_al

In [30]:
GGUF_F16  = "./mistral-7b-v0.1-f16.gguf"
GGUF_Q2   = "./mistral-7b-v0.1-Q2_K.gguf"
QUANT_TYPE = "Q2_K"

# I binari compilati con CMake si trovano in llama.cpp/build/bin/
!./llama.cpp/build/bin/llama-quantize {GGUF_F16} {GGUF_Q2} {QUANT_TYPE}

print(f"File quantizzato ({QUANT_TYPE}) generato: {GGUF_Q2}")

ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library
ggml_metal_library_compile_all: compiled 'fa' library in 0.007 sec
ggml_metal_library_compile_all: compiled 'mul_mv' library in 0.008 sec
ggml_metal_library_compile_all: compiled 'mul_mm' library in 0.009 sec
ggml_metal_library_compile_all: compiled 'quantize' library in 0.007 sec
ggml_metal_library_compile_all: compiled 'softmax' library in 0.001 sec
ggml_metal_library_compile_all: compiled 'norm' library in 0.001 sec
ggml_metal_library_compile_all: compiled 'unary' library in 0.001 sec
ggml_metal_library_compile_all: compiled 'binbcast' library in 0.001 sec
ggml_metal_library_compile_all: compiled 'reduce' library in 0.001 sec
ggml_metal_library_compile_all: compiled 'tri' library in 0.008 sec
ggml_metal_library_compile_all: compiled 'ssm' library in 0.001 sec
ggml_metal_library_compile_all: compiled 'wkv' library in 0.008 sec
ggml_metal_library_compile_all

---
## 5. Inferenza con llama-cpp-python

In [22]:
from llama_cpp import Llama

GGUF_Q4 = "./mistral-7b-v0.1-Q4_K_M.gguf"

llm = Llama(
    model_path=GGUF_Q4,
    n_ctx=4096,
    n_gpu_layers=0,   # imposta 32 per full GPU offload su CUDA
    verbose=False,
)

prompt = "La quantizzazione di un modello LLM consiste in"

output = llm(
    prompt,
    max_tokens=200,
    temperature=0.7,
    echo=True,
)

print(output["choices"][0]["text"])

La quantizzazione di un modello LLM consiste in troncare i pesi e le attivazioni dei neuroni del modello in un insieme finito di valori.

Questo consente di ridurre significativamente la memoria richiesta per il modello, ma anche di ottenere una maggiore robustezza nei confronti di errori di quantizzazione, e di migliorare le prestazioni del modello in termini di velocità e accuracy.

Il processo di quantizzazione coinvolge in genere la trasformazione dei valori di peso e attivazione in un formato a bassa precisione (ad esempio, float16), o la loro conversione in un insieme finito di valori discreti (ad esempio, 8-bit).

Questa tecnica è particolarmente utile per la quantizzazione in tempo reale, in cui è necessario esegu


In [32]:
from llama_cpp import Llama

GGUF_Q2 = "./mistral-7b-v0.1-Q2_K.gguf"

llm = Llama(
    model_path=GGUF_Q2,
    n_ctx=4096,
    n_gpu_layers=0,   # imposta 32 per full GPU offload su CUDA
    verbose=False,
)

prompt = "La quantizzazione di un modello LLM consiste in"

output = llm(
    prompt,
    max_tokens=200,
    temperature=0.7,
    echo=True,
)

print(output["choices"][0]["text"])

La quantizzazione di un modello LLM consiste in riparametrizzare il modello in modo da rendere la sua media di log-likelihood
GroupLayout

# 1 Introduction

La quantizzazione di un modello LLM consiste in riparametrizzare il modello in modo da rendere la sua media di log-likelihood

\[\begin{equation}\begin{aligned}\mathcal{L} = \mathbb{E}[\log P(y,z|x,w,b,c, \theta)] - \mathbb{E}[\log \pi(y|x,w,b,c, \theta)]\end{aligned}\end{equation}\]

e di minimizzare la sua varianza.

# 2 The case of a conditional model

Consider a conditional LLM whose data is \(y\), model is \(p\) and predictor is \(q\).

\


---
## 6. (Opzionale) Inferenza via terminale con llama-cli

Esegui il comando seguente direttamente da terminale (fuori da Jupyter):

```bash
./llama.cpp/llama-cli \
    -m ./Mistral-7B-v0.1/mistral-7b-v0.1-Q4_K_M.gguf \
    -p "La quantizzazione di un modello LLM consiste in" \
    -n 200 --temp 0.7 --ctx-size 4096
```

Oppure eseguilo dalla cella seguente:

In [ ]:
GGUF_Q4 = "./Mistral-7B-v0.1/mistral-7b-v0.1-Q4_K_M.gguf"

!./llama.cpp/llama-cli \
    -m {GGUF_Q4} \
    -p "La quantizzazione di un modello LLM consiste in" \
    -n 200 --temp 0.7 --ctx-size 4096